In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import json
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix


In [2]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

TRAIN_DIR = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\LWDCD\dataset_split\train"
VAL_DIR = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\LWDCD\dataset_split\val"
TEST_DIR = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\LWDCD\dataset_split\test"

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.15,
    shear_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

num_classes = train_generator.num_classes

Found 2351 images belonging to 3 classes.
Found 506 images belonging to 3 classes.
Found 505 images belonging to 3 classes.


In [4]:
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

In [5]:
inputs = layers.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = models.Model(inputs, outputs)
model.summary()

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        "VGG16_transfer_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)           │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ vgg16 (Functional)                   │ (None, 7, 7, 512)           │      14,714,688 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 512)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 512)                 │           2,048 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         131,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 3)                   │             771 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 14,848,835 (56.64 MB)

 Trainable params: 133,123 (520.01 KB)

 Non-trainable params: 14,715,712 (56.14 MB)

In [6]:
history_head = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,
    callbacks=callbacks
)

base_model.trainable = True

C:\Users\KristianHaltenJensen\anaconda4\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4044 - loss: 1.3586
Epoch 1: val_accuracy improved from -inf to 0.63834, saving model to VGG16_transfer_best.keras
74/74 ━━━━━━━━━━━━━━━━━━━━ 217s 3s/step - accuracy: 0.4054 - loss: 1.3563 - val_accuracy: 0.6383 - val_loss: 0.8636 - learning_rate: 1.0000e-04
Epoch 2/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6340 - loss: 0.8250
Epoch 2: val_accuracy improved from 0.63834 to 0.75692, saving model to VGG16_transfer_best.keras
74/74 ━━━━━━━━━━━━━━━━━━━━ 232s 3s/step - accuracy: 0.6341 - loss: 0.8253 - val_accuracy: 0.7569 - val_loss: 0.6464 - learning_rate: 1.0000e-04
Epoch 3/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.6873 - loss: 0.7639
Epoch 3: val_accuracy improved from 0.75692 to 0.79447, saving model to VGG16_transfer_best.keras
74/74 ━━━━━━━━━━━━━━━━━━━━ 261s 4s/step - accuracy: 0.6875 - loss: 0.7635 - val_accuracy: 0.7945 - val_loss: 0.5768 - learning_rate: 1.0000e-04
Epoch 4/50
74/74 ━━━━━━━━━━━

In [7]:
base_model.trainable = True

for layer in base_model.layers[:15]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,
    callbacks=callbacks
)

test_loss, test_acc = model.evaluate(test_generator)
print(f"Test accuracy: {test_acc:.4f}")

Epoch 1/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8968 - loss: 0.2884
Epoch 1: val_accuracy improved from 0.90711 to 0.91502, saving model to VGG16_transfer_best.keras
74/74 ━━━━━━━━━━━━━━━━━━━━ 194s 3s/step - accuracy: 0.8969 - loss: 0.2882 - val_accuracy: 0.9150 - val_loss: 0.2044 - learning_rate: 1.0000e-05
Epoch 2/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9055 - loss: 0.2500
Epoch 2: val_accuracy improved from 0.91502 to 0.94862, saving model to VGG16_transfer_best.keras
74/74 ━━━━━━━━━━━━━━━━━━━━ 192s 3s/step - accuracy: 0.9057 - loss: 0.2498 - val_accuracy: 0.9486 - val_loss: 0.1659 - learning_rate: 1.0000e-05
Epoch 3/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9216 - loss: 0.2078
Epoch 3: val_accuracy did not improve from 0.94862
74/74 ━━━━━━━━━━━━━━━━━━━━ 191s 3s/step - accuracy: 0.9216 - loss: 0.2078 - val_accuracy: 0.9427 - val_loss: 0.1508 - learning_rate: 1.0000e-05
Epoch 4/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9317 - lo

In [8]:


pred_probs = model.predict(test_generator)
y_pred = np.argmax(pred_probs, axis=1)
y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=class_names))
print(confusion_matrix(y_true, y_pred))

16/16 ━━━━━━━━━━━━━━━━━━━━ 36s 2s/step
                  precision    recall  f1-score   support

   Healthy Wheat       0.95      0.95      0.95       175
       Leaf Rust       0.97      0.95      0.96       190
Wheat Loose Smut       0.94      0.96      0.95       140

        accuracy                           0.95       505
       macro avg       0.95      0.95      0.95       505
    weighted avg       0.95      0.95      0.95       505

[[167   4   4]
 [  5 181   4]
 [  4   2 134]]


In [9]:

with open("VGG16_head_history_LWDCD.json", "w") as f:
    json.dump(history_head.history, f)

In [10]:
with open("VGG16_finetune_history_LWDCD.json", "w") as f:
    json.dump(history_fine.history, f)

In [11]:
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test accuracy: {test_acc:.4f}")

16/16 ━━━━━━━━━━━━━━━━━━━━ 34s 2s/step - accuracy: 0.9524 - loss: 0.1462
Test accuracy: 0.9545


In [12]:


pred_probs = model.predict(test_generator)
y_pred = np.argmax(pred_probs, axis=1)
y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=class_names))
print(confusion_matrix(y_true, y_pred))

16/16 ━━━━━━━━━━━━━━━━━━━━ 34s 2s/step
                  precision    recall  f1-score   support

   Healthy Wheat       0.95      0.95      0.95       175
       Leaf Rust       0.97      0.95      0.96       190
Wheat Loose Smut       0.94      0.96      0.95       140

        accuracy                           0.95       505
       macro avg       0.95      0.95      0.95       505
    weighted avg       0.95      0.95      0.95       505

[[167   4   4]
 [  5 181   4]
 [  4   2 134]]


In [13]:
model.save("VGG16_finetuned_LWDCD.keras")

In [14]:
model.save("VGG16_head_LWDCD.keras")